## Lesson Overview

**What this lesson teaches:** how to upload a reusable file with Anthropic's Files API, attach it to a request, let Claude analyze it with the code execution tool, and download a generated artifact.

**What's happening under the hood:**
1. Upload `streaming.csv` once and receive a persistent file ID.
2. Attach that file to Claude as a `container_upload` content block.
3. Give Claude the server-side code execution tool.
4. Let Claude inspect the CSV, calculate churn drivers, and create a plot.
5. Inspect typed response blocks and download any generated files.
6. List, inspect, and optionally delete uploaded files.

**You will build:** a reusable, opt-in workflow for analyzing a customer-streaming dataset and retrieving its generated chart.

> Files uploaded to the API are stored remotely until they expire or are deleted. Code runs in an isolated server-side container. Review sensitive data, retention requirements, and generated conclusions before using this workflow in production. Live cells may incur API charges.

# Lesson 18: Files API and Code Execution

The Files API separates file transfer from model requests. Instead of Base64-encoding the same dataset in every message, upload it once and refer to its file ID. With code execution enabled, Claude can load the attached file in a sandbox, use Python analysis libraries, and return both an explanation and downloadable artifacts.

```text
local CSV → Files API → file_id ─┐
                                  ├→ Claude + code execution → findings
analysis instructions ────────────┘                         └→ plot file
```

The uploaded source file and files generated inside the execution container have file IDs, but they play different roles: one is request input and the others are response artifacts.

## Setup

Install or upgrade the SDK once if needed:

```python
%pip install -U anthropic python-dotenv
```

Add `ANTHROPIC_API_KEY=...` to a `.env` file. The path lookup supports opening this notebook from either the repository root or `Claude_API_Training`.

This lesson uses beta API features. The required beta headers are set explicitly so it is clear which capabilities the request relies on.

In [1]:
import os
from pathlib import Path

try:
    from anthropic import Anthropic
except ImportError:
    Anthropic = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

for env_path in (Path('.env'), Path('Claude_API_Training/.env')):
    if env_path.exists():
        load_dotenv(env_path)
        break

api_key = os.getenv('ANTHROPIC_API_KEY')
client = (
    Anthropic(
        api_key=api_key,
        default_headers={
            'anthropic-beta': 'code-execution-2025-08-25,files-api-2025-04-14'
        },
    )
    if Anthropic is not None and api_key
    else None
)
model = 'claude-sonnet-4-5'

if Anthropic is None:
    print('Install or upgrade anthropic before running the live examples.')
elif client is None:
    print('Add ANTHROPIC_API_KEY to .env before running the live examples.')
else:
    print(f'Claude client ready. Model: {model}')

Claude client ready. Model: claude-sonnet-4-5


## Locate and Preview the Dataset

Always inspect a file locally before uploading it. This confirms the path, size, delimiter, and column names without sending data or spending API tokens. Reading only the first few lines avoids loading the full dataset.

In [2]:
dataset_candidates = (
    Path('streaming.csv'),
    Path('Claude_API_Training/streaming.csv'),
)
dataset_path = next((path for path in dataset_candidates if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError('Could not find streaming.csv.')

with dataset_path.open(encoding='utf-8') as dataset_file:
    preview_lines = [dataset_file.readline().rstrip() for _ in range(4)]

print(f'Dataset: {dataset_path} ({dataset_path.stat().st_size:,} bytes)')
print('\n'.join(preview_lines))

Dataset: streaming.csv (25,733 bytes)
UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned
USER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0
USER_00002,Premium,41.4,Drama,5,9,45.7,3,17.99,0
USER_00003,Standard,33.6,Action,1,7,32.3,4,12.99,1


## Files API Helpers

The upload endpoint accepts a filename, binary file object, and MIME type. Its response contains metadata such as the reusable `id` and filename. The remaining helpers expose the file lifecycle: list, retrieve metadata, download, and delete.

Deletion is intentionally not automatic. A learner may want to reuse the uploaded file in later requests, and remote deletion should always be deliberate.

In [3]:
MIME_TYPES = {
    '.csv': 'text/csv',
    '.json': 'application/json',
    '.pdf': 'application/pdf',
    '.txt': 'text/plain',
    '.md': 'text/plain',
    '.png': 'image/png',
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
}


def require_client():
    if client is None:
        raise RuntimeError('Complete the API setup before making a live request.')


def upload_file(path):
    require_client()
    path = Path(path)
    mime_type = MIME_TYPES.get(path.suffix.lower())
    if mime_type is None:
        raise ValueError(f'Unsupported file extension: {path.suffix}')

    with path.open('rb') as file_object:
        return client.beta.files.upload(
            file=(path.name, file_object, mime_type)
        )


def list_files():
    require_client()
    return client.beta.files.list()


def get_file_metadata(file_id):
    require_client()
    return client.beta.files.retrieve_metadata(file_id)


def download_file(file_id, destination):
    require_client()
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    client.beta.files.download(file_id).write_to_file(destination)
    return destination


def delete_file(file_id):
    require_client()
    return client.beta.files.delete(file_id)

## Upload the CSV

Uploading creates remote state and is therefore opt-in. Set `run_upload = True` when you are ready. Save the returned file ID; later cells attach it without uploading the CSV again.

If you restart the kernel, either upload again or assign a still-valid prior ID to `uploaded_file_id`.

In [4]:
run_upload = True
uploaded_file = None
uploaded_file_id = None

if run_upload and client is None:
    print('Complete the API setup before uploading the dataset.')
elif run_upload:
    uploaded_file = upload_file(dataset_path)
    uploaded_file_id = uploaded_file.id
    print('Uploaded:', uploaded_file.filename)
    print('File ID:', uploaded_file_id)
else:
    print('Upload disabled. Set run_upload = True when ready.')

Uploaded: streaming.csv
File ID: file_01P15DNzN9o7iV8e5rW7Ma1s


## Build the Analysis Request

A `container_upload` block makes the uploaded file available inside Claude's execution environment. It is different from a `document` block: the goal here is for code to open and analyze the raw CSV, not merely for the model to read a supported document inline.

The prompt asks for a reproducible sequence: audit the schema, define the churn target from observed values, calculate evidence, make a plot, and state limitations. It also reminds Claude that separate execution calls may start clean, so each call must load its own imports and data.

In [5]:
analysis_prompt = """
Analyze the attached streaming-customer CSV to identify the major drivers of churn.

Use code execution to:
1. Discover the uploaded filename and inspect its shape, columns, types, missing
   values, duplicates, and the observed values of the churn target.
2. State exactly how you define the positive churn class from the data.
3. Calculate overall churn and churn rates for important categorical segments.
4. Compare useful numeric features between churned and retained customers.
5. Rank the strongest associations carefully; do not describe correlation as
   causation or expose raw personal identifiers.
6. Create at least one clear, detailed plot of the most useful findings and save
   it as a PNG file for download.

Return a concise executive summary, supporting measurements, limitations, and
practical next analyses. Mention the generated plot filename.

Every code execution may begin with a clean state. In every execution, import all
required libraries, rediscover or reopen the uploaded file, and recreate variables.
""".strip()


def analysis_message(file_id):
    if not file_id:
        raise ValueError('A file ID is required. Run the upload cell first.')
    return {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': analysis_prompt},
            {'type': 'container_upload', 'file_id': file_id},
        ],
    }


print(analysis_prompt)

Analyze the attached streaming-customer CSV to identify the major drivers of churn.

Use code execution to:
1. Discover the uploaded filename and inspect its shape, columns, types, missing
   values, duplicates, and the observed values of the churn target.
2. State exactly how you define the positive churn class from the data.
3. Calculate overall churn and churn rates for important categorical segments.
4. Compare useful numeric features between churned and retained customers.
5. Rank the strongest associations carefully; do not describe correlation as
   causation or expose raw personal identifiers.
6. Create at least one clear, detailed plot of the most useful findings and save
   it as a PNG file for download.

Return a concise executive summary, supporting measurements, limitations, and
practical next analyses. Mention the generated plot filename.

Every code execution may begin with a clean state. In every execution, import all
required libraries, rediscover or reopen the uploade

## Local Request Check

Before a live call, use a placeholder ID to verify the message structure. This catches malformed content blocks without uploading data or invoking a model.

In [6]:
test_request = analysis_message('file_example123')
text_block, upload_block = test_request['content']

assert test_request['role'] == 'user'
assert text_block['type'] == 'text'
assert 'major drivers of churn' in text_block['text']
assert upload_block == {
    'type': 'container_upload',
    'file_id': 'file_example123',
}
assert MIME_TYPES[dataset_path.suffix.lower()] == 'text/csv'

print('Local file and request checks passed.')

Local file and request checks passed.


## Run Code Execution

The tool definition grants Claude access to Anthropic's managed code execution environment. Claude decides when and how many times to invoke it. The response may interleave explanatory text, tool calls, and tool results, so do not assume every content block has a `.text` attribute.

Set `run_analysis = True` after uploading the CSV. This is the main billable operation in the lesson.

In [7]:
CODE_EXECUTION_TOOL = {
    'type': 'code_execution_20250825',
    'name': 'code_execution',
}

run_analysis = True
analysis_response = None

if run_analysis and client is None:
    print('Complete the API setup before running the analysis.')
elif run_analysis and not uploaded_file_id:
    print('Upload the CSV first, or assign a valid prior ID to uploaded_file_id.')
elif run_analysis:
    analysis_response = client.messages.create(
        model=model,
        max_tokens=6000,
        messages=[analysis_message(uploaded_file_id)],
        tools=[CODE_EXECUTION_TOOL],
    )
    print('Stop reason:', analysis_response.stop_reason)
    print('Usage:', analysis_response.usage)
else:
    print('Analysis disabled. Set run_analysis = True when ready.')

Stop reason: end_turn
Usage: Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=186856, output_tokens=14188, server_tool_use=ServerToolUsage(web_fetch_requests=0, web_search_requests=0), service_tier='standard')


## Inspect Typed Response Blocks

A code-enabled response is structured, not just a string. Printing block types first makes the execution trace visible. Then extract only `text` blocks for the reader-facing report. Keep the complete response object if you need audit details or generated file references.

In [8]:
def text_from_message(message):
    return '\n'.join(
        block.text for block in message.content
        if block.type == 'text'
    )


if analysis_response is None:
    print('No live response yet.')
else:
    print('Block types:', [block.type for block in analysis_response.content])
    print('\n--- Final report ---\n')
    print(text_from_message(analysis_response))

Block types: ['text', 'server_tool_use', 'bash_code_execution_tool_result', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text']



## Find Generated File IDs

Generated artifacts are referenced inside nested tool-result data. SDK object shapes can evolve, so this helper converts each response block to ordinary Python data and recursively finds values whose key is `file_id`. It deduplicates IDs while preserving their first-seen order.

Inspect the surrounding response metadata before downloading: an analysis can generate more than one artifact, and an ID alone does not prove that the file is the requested plot.

In [9]:
def find_values(data, key):
    if isinstance(data, dict):
        for item_key, value in data.items():
            if item_key == key:
                yield value
            yield from find_values(value, key)
    elif isinstance(data, list):
        for value in data:
            yield from find_values(value, key)


def generated_file_ids(message):
    found = []
    for block in message.content:
        data = block.model_dump() if hasattr(block, 'model_dump') else block
        for file_id in find_values(data, 'file_id'):
            if isinstance(file_id, str) and file_id not in found:
                found.append(file_id)
    return found


artifact_ids = (
    generated_file_ids(analysis_response)
    if analysis_response is not None
    else []
)
print('Generated artifact IDs:', artifact_ids)

Generated artifact IDs: ['file_01Uq5XRab6rjMNz9pUwZSjbA', 'file_01BzkJVFm8xotM59gDpNTeHJ']


## Download a Generated Plot

Choose the appropriate artifact after inspecting the response, then download it to a new local path. The example defaults to the first discovered artifact and never uses a hard-coded ID from somebody else's run.

Set `run_download = True` only after confirming that `artifact_ids[0]` is the PNG you want. Existing files at the destination may be overwritten by the SDK.

In [11]:
run_download = True
downloaded_plot = True
plot_destination = dataset_path.parent / 'churn_drivers.png'

if run_download and not artifact_ids:
    print('No generated artifact ID is available. Run and inspect the analysis first.')
elif run_download:
    downloaded_plot = download_file(artifact_ids[0], plot_destination)
    print(f'Downloaded {artifact_ids[0]} to {downloaded_plot}')
else:
    print('Download disabled. Confirm the artifact ID, then set run_download = True.')

Downloaded file_01Uq5XRab6rjMNz9pUwZSjbA to churn_drivers.png


## Manage Uploaded Files

The Files API can list uploads and retrieve metadata without invoking Claude. This helps recover an ID after a kernel restart and verify a file before reuse. Listing may be paginated; this compact example displays the current page returned by the SDK.

The deletion cell is separately gated because it changes remote state and makes that file ID unusable.

In [ ]:
run_file_inspection = False

if run_file_inspection and client is None:
    print('Complete the API setup before inspecting remote files.')
elif run_file_inspection:
    page = list_files()
    for item in page.data:
        print(item.id, item.filename, item.size_bytes)

    if uploaded_file_id:
        print('Current upload metadata:', get_file_metadata(uploaded_file_id))
else:
    print('Remote inspection disabled. Set run_file_inspection = True when ready.')

In [ ]:
delete_uploaded_source = False

if delete_uploaded_source and not uploaded_file_id:
    print('No uploaded source file ID is available.')
elif delete_uploaded_source:
    deletion = delete_file(uploaded_file_id)
    print('Deleted remote source file:', uploaded_file_id)
    print(deletion)
else:
    print('Deletion disabled. Set delete_uploaded_source = True to remove the upload.')

## Troubleshooting

1. **`beta.files` is missing:** upgrade the `anthropic` package and restart the kernel.
2. **Authentication fails:** verify `ANTHROPIC_API_KEY` and which `.env` file was loaded.
3. **The CSV is not found:** run from the repository root or `Claude_API_Training`, or update `dataset_candidates`.
4. **The model cannot see the file:** confirm that the message uses `container_upload` and the ID belongs to an available upload.
5. **No plot ID appears:** inspect every response block, strengthen the instruction to save a PNG, and check whether execution ended because of a token or tool error.
6. **A download fails:** verify the artifact ID instead of reusing the example ID from another session.
7. **Analysis is inconsistent:** make the target definition and requested metrics explicit, then validate important calculations independently.
8. **The file is sensitive:** do not upload it until organizational policy, data minimization, and retention requirements permit it.

## Practice: Extend the Workflow

Try these one at a time and inspect both the response text and execution trace:

1. **Reuse the upload:** ask a second question with the same `uploaded_file_id` and no second upload.
2. **Improve validation:** request confidence intervals or minimum segment sizes for churn-rate comparisons.
3. **Create another artifact:** ask for a CSV table of ranked segments as well as the PNG plot, then download both IDs.
4. **Test robustness:** ask Claude to explain how missing values and high-cardinality columns affected the analysis.
5. **Audit the answer:** reproduce two reported statistics locally with pandas and compare them.
6. **Clean up:** list remote files, retrieve the source metadata, and deliberately delete the upload when it is no longer needed.

## Summary

- The Files API uploads data once and returns a reusable file ID.
- A `container_upload` content block places that file in the code execution environment.
- The code execution tool lets Claude inspect data, calculate metrics, and create artifacts in an isolated container.
- Code-enabled responses contain multiple typed blocks; extract text selectively and inspect tool results for artifact IDs.
- Download IDs from the current response rather than hard-coding IDs from an earlier run.
- Remote files have a lifecycle: upload, list or retrieve, reuse, download when applicable, and delete deliberately.
- Model-generated analysis and plots still require privacy review and independent validation.